# Which engine answered, and why

Mag$\nu$s does not have one algorithm. `oscprob.ENGINE_FAMILIES` registers **eight**, in
**five** families, and `strategy='auto'` picks between them per request.

Two other counts are in circulation, for different reasons and about different sets. The
companion paper's Fig. 1b draws the six *dispatch rows* a scenario wrapper walks, which omits
`constant` (folded into the energy-batched row) and `expm`. The `engines.rst` page counts
seven: it lists `constant` separately and says that `expm` never answers a request but serves
as an oracle. The dictionary printed below is the registry, which is the superset. Most of the time you neither know nor need to
know which one ran -- but when an answer looks wrong, "which engine produced this" is the first
question, and Mag$\nu$s will tell you.

The second half of this notebook is about something stronger. Every silently-wrong result the
package's adversarial validation ever found came from a method **certifying itself**: refining
its own knobs, comparing itself with itself, and agreeing. When a method has a blind spot both
sides of that comparison share it, and the agreement carries no information. Running two
genuinely *different* engines needs no oracle at all, and detects exactly the class that
self-certification cannot.

In [1]:
# The figures are set through LaTeX where one is available; where it is not,
# matplotlib's own mathtext renders the labels instead.  Same numbers either way.
import shutil

import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = shutil.which('latex') is not None

In [2]:
import warnings

import numpy as np

# Mag(nu)s is imported as an installed package -- from the repository root,
# 'pip install -e .' (add [plot] for magnus.plotting). No sys.path juggling.
import magnus.oscprob as oscprob
import magnus.matter as matter
import magnus.globaldefs as gd

NE0, L_SCALE = gd.NUM_DENSITY_E_SUN_CENTRAL, gd.L_SCALE_SUN
PARAMS_2NU = {'sth': 0.55, 'Dm2': 7.5e-5}
ne = matter.exp_density_profile(NE0, L_SCALE)
BASELINE = 1.0*L_SCALE

## 1. The engines, and which share machinery

The families matter more than the engines. Two engines in the same family share code and
assumptions, so their agreement is weak evidence; two in different families agreeing is
strong evidence.

In [3]:
for engine, family in oscprob.ENGINE_FAMILIES.items():
    print('%-12s %s' % (engine, family))

hybrid       adiabatic
ip_exp       interaction-picture
magnus       magnus-ladder
cumulative   magnus-ladder
separable    magnus-ladder
average      phase-average
expm         exact
constant     exact


## 2. What ran, for this request

Pass a dictionary as `strategy_info` and it comes back filled in. `engine` names what answered,
`family` places it, `certified` says whether that engine was able to vouch for its own result,
and `declined` lists what stood aside and why.

In [4]:
print('%-10s %-11s %-21s %-10s %s'
      % ('strategy', 'engine', 'family', 'certified', 'declined'))
print('-'*76)
for strategy in ('auto', 'hybrid', 'magnus'):
    info = {}
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        oscprob.osc_prob_matter_std_potential(
            2, ne, 10.0e6, BASELINE, PARAMS_2NU, L0=0.0,
            density_is_of_number_of_electrons=True,
            strategy=strategy, strategy_info=info)
    print('%-10s %-11s %-21s %-10s %s'
          % (strategy, info.get('engine'), info.get('family'),
             info.get('certified'), info.get('declined') or '--'))

strategy   engine      family                certified  declined
----------------------------------------------------------------------------
auto       magnus      magnus-ladder         None       [('hybrid', 'auto prefers the ladder')]
hybrid     hybrid      adiabatic             True       --


magnus     ip_exp      interaction-picture   None       --


On this smooth solar profile, at the default tolerance of $10^{-3}$, `'auto'` hands the
request to the **Magnus ladder**; `declined` says why. The adiabatic engine's search for
non-adiabatic windows costs the same at any tolerance, so at a loose one over a moderate phase
it is the slower route. Asked for by name, with `strategy='hybrid'`, it answers and certifies
itself. Forcing `strategy='magnus'` gets a third family, the interaction-picture integrator.
Neither it nor the ladder attempts to certify itself, which is why `certified` is `None`
rather than `False`.

## 3. Cross-checking: an error bar with no oracle

`cross_check_strategies` answers the same request with every engine that applies and reports
how far apart they are. It is never on by default -- it multiplies the cost of the call by the
number of engines -- and a large spread is *reported*, never raised. What it means depends on
the request, and deciding that is your job.

In [5]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    out = oscprob.cross_check_strategies(
        oscprob.osc_prob_matter_std_potential, 2, ne, 10.0e6, BASELINE,
        PARAMS_2NU, L0=0.0, density_is_of_number_of_electrons=True)

print('ran                  :', sorted(out['ran']))
print('families             :', sorted(set(out['families'].values())))
print('max spread           : %.3e' % out['max_spread'])
print('max across families  : %.3e  %s'
      % (out['max_spread_independent'], out['max_spread_independent_pair']))
print()
for label, reason in out['declined'].items():
    print('declined %-10s %s' % (label, reason))

ran                  : ['cumulative', 'hybrid', 'ip_exp', 'magnus']
families             : ['adiabatic', 'interaction-picture', 'magnus-ladder']
max spread           : 2.243e-03
max across families  : 2.234e-03  ('hybrid', 'magnus')

declined separable  does not apply to this request (answered by magnus)
declined constant   does not apply to this request (answered by magnus)
declined expm       H varies with position on [0, 3.34499e+14]; expm is exact only for a piecewise-constant H with declared edges


Four engines, three families, and they agree to $2\times10^{-3}$ with no reference solution
anywhere in sight. That number is a far more honest error bar than any `rtol` (notebook 21),
because the things being compared do not share a method.

## 4. The case it exists for

Now a density profile with an **undeclared step** in it -- the construction from the package's
adversarial-validation findings. This is where the adiabatic engine historically returned a
confidently wrong answer.

In [6]:
def step_ne(l):
    """A density jump the caller does not declare."""
    x = np.asarray(l, dtype=float)
    out = np.where(x < 0.5*BASELINE, 0.02*NE0, 0.30*NE0)
    return out[()] if out.ndim == 0 else out

print('%-10s %-11s %-11s %-11s %s'
      % ('strategy', 'P_ee', 'engine', 'certified', 'warnings'))
print('-'*74)
for strategy in ('auto', 'hybrid'):
    info = {}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        P = oscprob.osc_prob_matter_std_potential(
            2, step_ne, 50.0e6, BASELINE, PARAMS_2NU, L0=0.0,
            density_is_of_number_of_electrons=True,
            strategy=strategy, strategy_info=info)
    names = sorted({w.category.__name__.replace('Warning', '') for w in caught})
    print('%-10s %-11.6f %-11s %-11s %s'
          % (strategy, float(np.asarray(P)[0][0]), info.get('engine'),
             info.get('certified'), ', '.join(names)))
    if info.get('declined'):
        print('%-10s   declined: %s' % ('', info['declined']))

strategy   P_ee        engine      certified   warnings
--------------------------------------------------------------------------
auto       0.079745    magnus      None        UnmarkedDiscontinuity
             declined: [('hybrid', 'the profile is not resolved at the probe scale')]
hybrid     0.491725    hybrid      False       HybridCertification, UnmarkedDiscontinuity


Two different answers, 0.080 and 0.492, from the same request. The package is no
longer silent about it in either direction:

* under `'auto'` the adiabatic engine **declines** -- "the profile is not resolved at the probe
  scale" -- and the Magnus ladder answers instead;
* forced with `strategy='hybrid'` it still answers, but reports `certified=False` and raises
  `HybridCertificationWarning`.

Both paths also raise `UnmarkedDiscontinuityWarning`, which names the actual fix: declare the
jump with `t_breakpoints`.

And the cross-check sees the disagreement without being told any of that:

In [7]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    out_step = oscprob.cross_check_strategies(
        oscprob.osc_prob_matter_std_potential, 2, step_ne, 50.0e6, BASELINE,
        PARAMS_2NU, L0=0.0, density_is_of_number_of_electrons=True)

for engine in sorted(out_step['ran']):
    print('%-12s (%-14s) P_ee = %.6f'
          % (engine, oscprob.ENGINE_FAMILIES[engine],
             float(np.asarray(out_step['answers'][engine])[0][0])))
print('\nmax spread across families: %.3e  %s'
      % (out_step['max_spread_independent'], out_step['max_spread_independent_pair']))

cumulative   (magnus-ladder ) P_ee = 0.079745
hybrid       (adiabatic     ) P_ee = 0.491725
magnus       (magnus-ladder ) P_ee = 0.079181

max spread across families: 4.125e-01  ('hybrid', 'magnus')


A spread of **0.41** on a probability. No ground truth was computed, no reference
code was installed, and nothing had to know in advance what was wrong with the profile. Two
engines from different families simply disagreed, which is all the signal you need to stop
trusting the number.

## 5. A zero spread is not always agreement

One trap, and it is the reason this function warns. `max_spread` is a plain float, and it is
`0.0` both when the engines agree perfectly and when **nothing was compared**. The commonest
way to reach the second case is to pass an entry point that has no `strategy` parameter --
`osc_prob` itself is one, and it is the function most of these notebooks call directly.

In [8]:
import magnus.hamiltonians as hamiltonians

h_vac = np.asarray(hamiltonians.hamiltonian_2nu_vacuum_energy_independent(**PARAMS_2NU))
vcc = matter.vcc_func_from_rho_func(ne, density_is_of_number_of_electrons=True)

def H_func(l):
    return h_vac/10.0e6 + np.diag([vcc(l), 0.0])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    empty = oscprob.cross_check_strategies(oscprob.osc_prob, H_func, 0.0, BASELINE)

print('ran        :', empty['ran'])
print('max_spread : %.1f   <- and yet nothing was compared' % empty['max_spread'])
print()
for w in caught:
    if w.category is oscprob.CrossCheckInconclusiveWarning:
        print('%s raised.' % w.category.__name__)

ran        : ()
max_spread : 0.0   <- and yet nothing was compared

CrossCheckInconclusiveWarning raised.


The same vacuous zero appears when only one engine runs, and
`max_spread_independent` is zero whenever every engine that ran belongs to a single family --
which is precisely the self-certification the cross-check exists to avoid.
`CrossCheckInconclusiveWarning` covers all three.

## Summary

| question | how to answer it |
|---|---|
| which engine ran? | pass `strategy_info={}` and read `engine` |
| do they share machinery? | `oscprob.ENGINE_FAMILIES` |
| did the engine vouch for itself? | `strategy_info['certified']` |
| how far apart are independent methods? | `cross_check_strategies(...)['max_spread_independent']` |
| why did an engine stand aside? | `strategy_info['declined']`, or `out['declined']` |

**Always check `ran` before reading a spread.** A cross-check that ran nothing reports perfect
agreement, and a cross-check confined to one family reports independent agreement it never
tested. The warning will tell you, but the number will not.

---

**Previous:** [What rtol and atol promise](21_magnus_what_tolerance_means.ipynb)  
**Next:** [When averaging rescues you](23_magnus_when_averaging_helps.ipynb) --- phase error falls away, envelope error does not  
[API reference](https://mbustama.github.io/Magnus/functions.html) &middot; [Documentation](https://mbustama.github.io/Magnus/) &middot; [All notebooks](.)